In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no cuda")


True
Tesla T4


In [2]:

#项目导入


%cd /content
!git clone https://github.com/Change-99/GraphMMI.git

/content
Cloning into 'GraphMMI'...
remote: Enumerating objects: 505, done.
remote: Counting objects: 100% (505/505), done.
remote: Compressing objects: 100% (376/376), done.
remote: Total 505 (delta 100), reused 499 (delta 94), pack-reused 0 (from 0)
Receiving objects: 100% (505/505), 17.37 MiB | 33.37 MiB/s, done.
Resolving deltas: 100% (100/100), done.


In [3]:
%cd /content/GraphMMI/GraphMMI/
! ls
! pwd


/content/GraphMMI/GraphMMI
exp  requirements.txt  runs  scripts  src
/content/GraphMMI/GraphMMI


In [4]:
#数据准备
from google.colab import drive
drive.mount('/content/drive')

%cd /content/GraphMMI/GraphMMI/
!mkdir -p data/processed/graph/
#!cp -r "/content/drive/MyDrive/data/processed/graph/random" /content/GraphMMI/GraphMMI/data/processed/graph/
!cp -r "/content/drive/MyDrive/data/external" /content/GraphMMI/GraphMMI/data


Mounted at /content/drive
/content/GraphMMI/GraphMMI


rm: invalid option -- '/'
Try 'rm --help' for more information.


In [7]:
#冒烟训练graphsage模型

!python -u scripts/train_gnn_transfer.py \
  --device cuda \
  --species worm cow \
  --encoders graphsage \
  --settings zero_shot \
  --epochs 1 \
  --neg-ratio 0.2 \
  --eval-neg-ratio 0.2 \
  --run-root runs/colab_smoke


[graphsage/zero_shot/source/worm] epoch=001 loss=0.6962 val_aupr=0.8595 val_auc=0.5012
[graphsage/zero_shot/source/cow] epoch=001 loss=0.6749 val_aupr=0.8620 val_auc=0.5526
[RESULT] graphsage/zero_shot worm->worm AUC=0.5922 AUPR=0.8934 F1=0.9071 MCC=0.1005 thr=0.5248
[RESULT] graphsage/zero_shot worm->cow AUC=0.5291 AUPR=0.8476 F1=0.1321 MCC=0.0276 thr=0.5574
[RESULT] graphsage/zero_shot cow->worm AUC=0.5700 AUPR=0.8834 F1=0.1292 MCC=0.1105 thr=0.5570
[RESULT] graphsage/zero_shot cow->cow AUC=0.5513 AUPR=0.8567 F1=0.6353 MCC=0.0813 thr=0.5580
Saved GNN transfer run to: runs/colab_smoke/20260514-115231


In [ ]:
#训练graphsage模型
!python -u scripts/train_gnn_transfer.py \
  --device cuda \
  --encoders graphsage \
  --settings zero_shot finetune \
  --epochs 40 \
  --finetune-epochs 15 \
  --run-root runs/gnn_transfer_colab \
  2>&1 | tee train_gnn_transfer.log


In [ ]:
#训练gatv2模型
!python -u scripts/train_gnn_transfer.py \
  --device cuda \
  --encoders gatv2 \
  --settings zero_shot finetune \
  --epochs 40 \
  --finetune-epochs 15 \
  --run-root runs/gnn_transfer_colab \
  2>&1 | tee train_gnn_transfer.log


In [8]:
##保存runs数据
SRC = "/content/GraphMMI/GraphMMI/runs"
DST = "/content/drive/MyDrive/runs"
!rm -rf "$DST"
!cp -r "$SRC" "$DST"


In [15]:
%cd /content/GraphMMI
!git pull




/content/GraphMMI
remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 10 (delta 6), reused 10 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 4.54 KiB | 2.27 MiB/s, done.
From https://github.com/Change-99/GraphMMI
   7976098..7614d5f  main       -> origin/main
Updating 7976098..7614d5f
Fast-forward
 GraphMMI/scripts/preprocess_graph_data.py | 212 ++++++++++++++++++++++++++++++
 GraphMMI/scripts/train_gnn_transfer.py    |  22 +++-
 GraphMMI/src/graphmmi/data.py             |  21 +++
 GraphMMI/src/graphmmi/models.py           |  26 +++-
 4 files changed, 272 insertions(+), 9 deletions(-)


In [ ]:
#优化后的训练命令

!python -u scripts/train_gnn_transfer.py \
  --device cuda \
  --encoders graphsage gatv2 \
  --settings strict_zero_shot calibrated_zero_shot finetune \
  --neg-strategy degree_aware \
  --eval-neg-strategy same \
  --finetune-strategy decoder \
  --residual \
  --layer-norm \
  --decoder-layer-norm \
  --epochs 40 \
  --finetune-epochs 15 \
  --run-root runs/gnn_transfer_optimized


In [17]:
%cd /content/GraphMMI/GraphMMI
!pwd
!ls


/content/GraphMMI/GraphMMI
/content/GraphMMI/GraphMMI
data		  scripts			train_gatv2_noid_full.log
requirements.txt  src
runs		  train_gatv2_noid_decoder.log


In [9]:
#同步数据到drive
from google.colab import drive
drive.mount('/content/drive')

SRC = "/content/GraphMMI/GraphMMI/runs"
DST = "/content/drive/MyDrive/runs"

!mkdir -p "$DST"
!rsync -av --delete "$SRC"/ "$DST"/

!echo "Synced Colab runs to Drive:"
!find "$DST" -maxdepth 2 -type f | head -30


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
sending incremental file list
deleting gnn_transfer_sim_smoke/20260514-100343/heatmaps/graphsage_zero_shot_mcc.png
deleting gnn_transfer_sim_smoke/20260514-100343/heatmaps/graphsage_zero_shot_f1.png
deleting gnn_transfer_sim_smoke/20260514-100343/heatmaps/graphsage_zero_shot_aupr.png
deleting gnn_transfer_sim_smoke/20260514-100343/heatmaps/graphsage_zero_shot_auc.png
deleting gnn_transfer_sim_smoke/20260514-100343/heatmaps/graphsage_zero_shot_all_metrics.png
deleting gnn_transfer_sim_smoke/20260514-100343/heatmaps/graphsage_zero_shot_acc.png
deleting gnn_transfer_sim_smoke/20260514-100343/heatmaps/
deleting gnn_transfer_sim_smoke/20260514-100343/transfer_metrics.json
deleting gnn_transfer_sim_smoke/20260514-100343/transfer_metrics.csv
deleting gnn_transfer_sim_smoke/20260514-100343/config.json
deleting gnn_transfer_sim_smoke/20260514-100343/
deleting gnn_tran

In [5]:
# 先重新预处理（生成带相似边的 .pt）
!python scripts/preprocess_graph_data.py --mirna-sim-edges --mrna-sim-edges

# # A. 原始二部图
# !python scripts/train_gnn_transfer.py --encoders graphsage gatv2

# # B. + miRNA-miRNA only
#!python scripts/train_gnn_transfer.py --encoders graphsage gatv2 --mirna-sim-edges

# # C. + mRNA-mRNA only
# !python scripts/train_gnn_transfer.py --encoders graphsage gatv2 --mrna-sim-edges

# # D. + both
# !python scripts/train_gnn_transfer.py --encoders graphsage gatv2 --mirna-sim-edges --mrna-sim-edges


human: nodes=5634 pos_edges=8909 train/val/test={'train': 6414, 'val': 713, 'test': 1782}
cow: nodes=3499 pos_edges=8534 train/val/test={'train': 6144, 'val': 683, 'test': 1707}
mouse: nodes=7034 pos_edges=16715 train/val/test={'train': 12035, 'val': 1337, 'test': 3343}
worm: nodes=1426 pos_edges=2098 train/val/test={'train': 1510, 'val': 168, 'test': 420}
Saved graph preprocessing outputs to: /content/GraphMMI/GraphMMI/data/processed/graph/random


In [23]:
!python -u scripts/train_gnn_transfer.py \
  --device cuda \
  --encoders graphsage gatv2 \
  --settings zero_shot finetune \
  --epochs 40 \
  --finetune-epochs 15 \
  --mirna-sim-edges \
  --mrna-sim-edges \
  --run-root runs/gnn_transfer_colab \
  2>&1 | tee train_gnn_transfer.log


[graphsage/zero_shot/source/human] epoch=001 loss=0.6926 val_aupr=0.6340 val_auc=0.6513
[graphsage/zero_shot/source/human] epoch=002 loss=0.6909 val_aupr=0.6618 val_auc=0.6822
[graphsage/zero_shot/source/cow] epoch=001 loss=0.6936 val_aupr=0.7017 val_auc=0.7239
[graphsage/zero_shot/source/mouse] epoch=001 loss=0.6948 val_aupr=0.6744 val_auc=0.6886
[graphsage/zero_shot/source/mouse] epoch=002 loss=0.6913 val_aupr=0.6965 val_auc=0.7135
[graphsage/zero_shot/source/mouse] epoch=003 loss=0.6877 val_aupr=0.7027 val_auc=0.7228
[graphsage/zero_shot/source/worm] epoch=001 loss=0.6932 val_aupr=0.6951 val_auc=0.6909
[graphsage/zero_shot/source/worm] epoch=002 loss=0.6896 val_aupr=0.7110 val_auc=0.7111
[RESULT] graphsage/zero_shot human->human AUC=0.6731 AUPR=0.6449 F1=0.6538 MCC=0.2695 thr=0.5077
[RESULT] graphsage/zero_shot human->cow AUC=0.7279 AUPR=0.7151 F1=0.6319 MCC=0.3352 thr=0.5108
[RESULT] graphsage/zero_shot human->mouse AUC=0.6988 AUPR=0.6652 F1=0.6636 MCC=0.2977 thr=0.5114
[RESULT] gr

In [8]:
!python -u scripts/train_gnn_transfer_v2.py \
  --device cuda \
  --species human cow mouse worm \
  --encoders graphsage \
  --settings zero_shot finetune \
  --epochs 40 \
  --finetune-epochs 15 \
  --patience 8 \
  --num-layers 4 \
  --mirna-sim-edges \
  --mrna-sim-edges \
  --run-root runs/gnn_transfer_final_v2 \
  2>&1 | tee train_gnn_transfer_v2.log


[graphsage/zero_shot/source/human] epoch=001 loss=0.6987 val_aupr=0.6161 val_auc=0.6410
[graphsage/zero_shot/source/human] epoch=002 loss=0.6950 val_aupr=0.6407 val_auc=0.6646
[graphsage/zero_shot/source/human] epoch=003 loss=0.6943 val_aupr=0.6612 val_auc=0.6801
[graphsage/zero_shot/source/human] epoch=004 loss=0.6909 val_aupr=0.6650 val_auc=0.6821
[graphsage/zero_shot/source/human] epoch=005 loss=0.6894 val_aupr=0.6669 val_auc=0.6845
[graphsage/zero_shot/source/human] epoch=006 loss=0.6892 val_aupr=0.6720 val_auc=0.6915
[graphsage/zero_shot/source/human] epoch=007 loss=0.6868 val_aupr=0.6779 val_auc=0.6990
[graphsage/zero_shot/source/cow] epoch=001 loss=0.6977 val_aupr=0.5747 val_auc=0.6093
[graphsage/zero_shot/source/cow] epoch=002 loss=0.6941 val_aupr=0.6210 val_auc=0.6581
[graphsage/zero_shot/source/cow] epoch=003 loss=0.6931 val_aupr=0.6569 val_auc=0.6889
[graphsage/zero_shot/source/cow] epoch=004 loss=0.6903 val_aupr=0.6808 val_auc=0.7034
[graphsage/zero_shot/source/cow] epoch=0

In [7]:
!git pull

remote: Enumerating objects: 240, done.
remote: Counting objects: 100% (240/240), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 220 (delta 139), reused 220 (delta 139), pack-reused 0 (from 0)
Receiving objects: 100% (220/220), 55.97 KiB | 9.33 MiB/s, done.
Resolving deltas: 100% (139/139), completed with 5 local objects.
From https://github.com/Change-99/GraphMMI
   266becc..7a72b30  main       -> origin/main
Updating 266becc..7a72b30
Fast-forward
 .../result/20260515-101848/config.json             |  53 ++
 .../result/20260515-101848/transfer_metrics.csv    |   2 +
 .../result/20260515-101848/transfer_metrics.json   |  32 +
 .../result/20260515-101948/config.json             |  53 ++
 .../result/20260515-101948/transfer_metrics.csv    |   2 +
 .../result/20260515-101948/transfer_metrics.json   |  32 +
 .../result/20260515-102046/config.json             |  53 ++
 .../result/20260515-102046/transfer_metrics.csv    |   2 +
 .../result/20260515-102046/transfer_metri